In [39]:
import pandas as pd
df = pd.read_csv("mrw_1992_appendix.csv")
df.head()

,No,Country,Non_oil,Intermediate,OECD,GDP_1960,GDP_1985,GDP_growth_pct,Working_age_growth_pct,Investment_pct_GDP,School_pct
0,1,Algeria,1,1,0,2485.0,4371.0,4.8,2.6,24.1,4.5
1,2,Angola,1,0,0,1588.0,1171.0,0.8,2.1,5.8,1.8
2,3,Benin,1,0,0,1116.0,1071.0,2.2,2.4,10.8,1.8
3,4,Botswana,1,1,0,959.0,3671.0,8.6,3.2,28.3,2.9
4,5,Burkina Faso,1,0,0,529.0,857.0,2.9,0.9,12.7,0.4


In [40]:
print(df.shape)
print(df.columns.tolist())

(121, 11)
['No', 'Country', 'Non_oil', 'Intermediate', 'OECD', 'GDP_1960', 'GDP_1985', 'GDP_growth_pct', 'Working_age_growth_pct', 'Investment_pct_GDP', 'School_pct']


In [42]:
non_oil = df[df["Non_oil"] == 1].copy()
intermediate = df[df["Intermediate"] == 1].copy()
oecd = df[df["OECD"] == 1].copy()

print("Non-oil sample:",len(non_oil))
print("Intermediate sample:", len(intermediate))
print("OECD sample:", len(oecd))

Non-oil sample: 98
Intermediate sample: 75
OECD sample: 22


In [43]:
import numpy as np
df["ln_GDP_1985"] = np.log(df["GDP_1985"])
df["ln_investment"] = np.log(df["Investment_pct_GDP"]/100)
df["ln_ngd"] = np.log(df["Working_age_growth_pct"]/100 + 0.05)


In [44]:
df[["Country", "ln_GDP_1985", "ln_investment", "ln_ngd"]].head()

,Country,ln_GDP_1985,ln_investment,ln_ngd
0,Algeria,8.382747,-1.422958,-2.577022
1,Angola,7.065613,-2.847312,-2.645075
2,Benin,6.976348,-2.225624,-2.603690
3,Botswana,8.208219,-1.262308,-2.501036
4,Burkina Faso,6.753438,-2.063568,-2.830218


In [45]:
non_oil = df[df["Non_oil"] == 1].copy()
intermediate = df[df["Intermediate"] == 1].copy()
oecd = df[df["OECD"] == 1].copy()

print("Non-oil:", len(non_oil))
print("Intermediate:", len(intermediate))
print("OECD", len(oecd))


Non-oil: 98
Intermediate: 75
OECD 22


In [46]:
df["ln_school"] = np.log(df["School_pct"]/100)
df[["Country", "ln_school"]].head()


,Country,ln_school
0,Algeria,-3.101093
1,Angola,-4.017384
2,Benin,-4.017384
3,Botswana,-3.540459
4,Burkina Faso,-5.521461


In [47]:
df[["Country", "ln_GDP_1985", "ln_investment", "ln_ngd", "ln_school"]].head()

,Country,ln_GDP_1985,ln_investment,ln_ngd,ln_school
0,Algeria,8.382747,-1.422958,-2.577022,-3.101093
1,Angola,7.065613,-2.847312,-2.645075,-4.017384
2,Benin,6.976348,-2.225624,-2.603690,-4.017384
3,Botswana,8.208219,-1.262308,-2.501036,-3.540459
4,Burkina Faso,6.753438,-2.063568,-2.830218,-5.521461


In [48]:

# UM of non oil
import statsmodels.api as sm
non_oil = df[df["Non_oil"] == 1].copy()
y = non_oil["ln_GDP_1985"]
X = non_oil[["ln_investment", "ln_ngd"]]
X = sm.add_constant(X)
model = sm.OLS(y,X).fit()
print(model.summary())

# RM of non oil
non_oil["restricted_x"] = (non_oil["ln_investment"] - non_oil["ln_ngd"])
non_oil[["Country", "restricted_x"]].head()
y_r = non_oil["ln_GDP_1985"]
X_r = non_oil[["restricted_x"]]
X_r = sm.add_constant(X_r)
model_r = sm.OLS(y_r, X_r).fit()
print(model_r.summary())

# F test
f_test = model.f_test("ln_investment + ln_ngd = 0")
print(f_test)
print("F-statistic:", float(f_test.fvalue))
print("P-value:", float(f_test.pvalue))

# alpha and se
b1 = model_r.params["restricted_x"]
alpha = b1 / (1+b1)
print("Estimated coefficient:", b1)

se_b1 = model_r.bse["restricted_x"]
se_alpha = se_b1 / (1+b1)**2
print("Implied alpha:", round(alpha, 4))
print("Standard error of alpha:", round(se_alpha, 4))


                            OLS Regression Results                            
Dep. Variable:            ln_GDP_1985   R-squared:                       0.601
Model:                            OLS   Adj. R-squared:                  0.592
Method:                 Least Squares   F-statistic:                     71.51
Date:                Thu, 29 Oct 2026   Prob (F-statistic):           1.13e-19
Time:                        23:15:34   Log-Likelihood:                -101.04
No. Observations:                  98   AIC:                             208.1
Df Residuals:                      95   BIC:                             215.8
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             5.4299      1.584      3.428

In [49]:
# UM of intermediate
intermediate = df[df["Intermediate"] == 1].copy()
y_int = intermediate["ln_GDP_1985"]
X_int = intermediate[["ln_investment", "ln_ngd"]]
X_int = sm.add_constant(X_int)
model_int = sm.OLS(y_int,X_int).fit()
print(model_int.summary())

# RM of intermediated
intermediate["restricted_x"] = (intermediate["ln_investment"] - intermediate["ln_ngd"])
intermediate[["Country", "restricted_x"]].head()
y_int_r = intermediate["ln_GDP_1985"]
X_int_r = intermediate[["restricted_x"]]
X_int_r = sm.add_constant(X_int_r)
model_int_r = sm.OLS(y_int_r, X_int_r).fit()
print(model_int_r.summary())

# F test
f_test_int = model_int.f_test("ln_investment + ln_ngd = 0")
print(f_test_int)
print("F-statistic:", float(f_test_int.fvalue))
print("P-value:", float(f_test_int.pvalue))

# alpha and se
b1_int = model_int_r.params["restricted_x"]
alpha_int = b1_int / (1+b1_int)
print("Estimated coefficient:", b1_int)

se_b1_int = model_int_r.bse["restricted_x"]
se_alpha_int = se_b1_int / (1+b1_int)**2
print("Implied alpha_int:", round(alpha_int, 4))
print("Standard error of alpha_int:", round(se_alpha_int, 4))

                            OLS Regression Results                            
Dep. Variable:            ln_GDP_1985   R-squared:                       0.599
Model:                            OLS   Adj. R-squared:                  0.588
Method:                 Least Squares   F-statistic:                     53.76
Date:                Thu, 29 Oct 2026   Prob (F-statistic):           5.21e-15
Time:                        23:15:44   Log-Likelihood:                -67.896
No. Observations:                  75   AIC:                             141.8
Df Residuals:                      72   BIC:                             148.7
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             5.3459      1.543      3.464

In [50]:
# UM of oecd
oecd = df[df["OECD"] == 1].copy()
y_oecd = oecd["ln_GDP_1985"]
X_oecd = oecd[["ln_investment", "ln_ngd"]]
X_oecd = sm.add_constant(X_oecd)
model_oecd = sm.OLS(y_oecd,X_oecd).fit()
print(model_oecd.summary())

#RM of oecd
oecd["restricted_x"] = (oecd["ln_investment"] - oecd["ln_ngd"])
oecd[["Country", "restricted_x"]].head()
y_oecd_r = oecd["ln_GDP_1985"]
X_oecd_r = oecd[["restricted_x"]]
X_oecd_r = sm.add_constant(X_oecd_r)
model_oecd_r = sm.OLS(y_oecd_r, X_oecd_r).fit()
print(model_oecd_r.summary())

# F test
f_test_oecd = model_oecd.f_test("ln_investment + ln_ngd = 0")
print(f_test_oecd)
print("F-statistic:", float(f_test_oecd.fvalue))
print("P-value:", float(f_test_oecd.pvalue))

# alpha and se
b1_oecd = model_oecd_r.params["restricted_x"]
alpha_oecd = b1_oecd / (1+b1_oecd)
print("Estimated coefficient:", b1_oecd)
se_b1_oecd = model_oecd_r.bse["restricted_x"]
se_alpha_oecd = se_b1_oecd / (1+b1_oecd)**2
print("Implied alpha_oecd:", round(alpha_oecd, 4))
print("Standard error of alpha_oecd:", round(se_alpha_oecd, 4))


                            OLS Regression Results                            
Dep. Variable:            ln_GDP_1985   R-squared:                       0.106
Model:                            OLS   Adj. R-squared:                  0.012
Method:                 Least Squares   F-statistic:                     1.126
Date:                Thu, 29 Oct 2026   Prob (F-statistic):              0.345
Time:                        23:15:49   Log-Likelihood:                -8.1659
No. Observations:                  22   AIC:                             22.33
Df Residuals:                      19   BIC:                             25.60
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             8.0206      2.518      3.185

In [58]:
# Augumented Solow Model 
# UM of non oil
non_oil = df[df["Non_oil"] == 1].copy()
y_Au = non_oil["ln_GDP_1985"]
X_Au = non_oil[["ln_investment", "ln_ngd", "ln_school"]]
X_Au = sm.add_constant(X_Au)
model_Au = sm.OLS(y_Au,X_Au).fit()
print(model_Au.summary())

# RM of non oil
non_oil["restricted_k"] = (non_oil["ln_investment"] - non_oil["ln_ngd"])
non_oil["restricted_h"] = (non_oil["ln_school"] - non_oil["ln_ngd"])

y_Au_r = non_oil["ln_GDP_1985"]
X_Au_r = non_oil[["restricted_k", "restricted_h"]]
X_Au_r = sm.add_constant(X_Au_r)
model_Au_r = sm.OLS(y_Au_r, X_Au_r).fit()
print(model_Au_r.summary())

# F test
f_test = model_Au.f_test("ln_investment + ln_ngd + ln_school = 0")
print(f_test)
print("F-statistic:", float(f_test.fvalue))
print("P-value:", float(f_test.pvalue))

# alpha and beta
b1 = model_Au_r.params["restricted_k"]
b2 = model_Au_r.params["restricted_h"]

alpha = b1 / (1+b1+b2)
beta = b2 / (1+b1+b2)

print("physical capital coefficient:", b1)
print("human capital coefficient:", b2)
print("Implied alpha:", round(alpha, 4))
print("Implied beta:", round(beta, 4))

# se of alpha and beta
cov_matrix = model_Au_r.cov_params().loc[["restricted_k", "restricted_h"], ["restricted_k", "restricted_h"]].to_numpy()
grad_alpha = np.array([(1 + b2) / (1+b1+b2)**2, -b1 / (1+b1+b2)**2])
grad_beta = np.array([-b2 / (1+b1+b2)**2,(1 + b1) / (1+b1+b2)**2])
se_alpha = np.sqrt(grad_alpha @ cov_matrix @ grad_alpha)
se_beta = np.sqrt(grad_beta @ cov_matrix @ grad_beta)

print("se of alpha:", round(se_alpha, 4))
print("se of beta:", round(se_beta, 4))



                            OLS Regression Results                            
Dep. Variable:            ln_GDP_1985   R-squared:                       0.786
Model:                            OLS   Adj. R-squared:                  0.779
Method:                 Least Squares   F-statistic:                     114.8
Date:                Thu, 29 Oct 2026   Prob (F-statistic):           2.54e-31
Time:                        23:47:33   Log-Likelihood:                -70.576
No. Observations:                  98   AIC:                             149.2
Df Residuals:                      94   BIC:                             159.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             6.8444      1.177      5.813

In [60]:
# Augumented Solow model
# UM of intermediate
intermediate = df[df["Intermediate"] == 1].copy()
y_int_Au = intermediate["ln_GDP_1985"]
X_int_Au = intermediate[["ln_investment", "ln_ngd", "ln_school"]]
X_int_Au = sm.add_constant(X_int_Au)
model_int_Au = sm.OLS(y_int_Au,X_int_Au).fit()
print(model_int_Au.summary())

# RM of intermediated
intermediate["restricted_k"] = (intermediate["ln_investment"] - intermediate["ln_ngd"])
intermediate["restricted_h"] = (intermediate["ln_school"] - intermediate["ln_ngd"])
y_int_Au_r = intermediate["ln_GDP_1985"]
X_int_Au_r = intermediate[["restricted_k", "restricted_h"]]
X_int_Au_r = sm.add_constant(X_int_Au_r)
model_int_Au_r = sm.OLS(y_int_Au_r, X_int_Au_r).fit()
print(model_int_Au_r.summary())

# F test
f_test = model_int_Au.f_test("ln_investment + ln_ngd + ln_school = 0")
print(f_test)
print("F-statistic:", float(f_test.fvalue))
print("P-value:", float(f_test.pvalue))

# alpha and beta
b1 = model_int_Au_r.params["restricted_k"]
b2 = model_int_Au_r.params["restricted_h"]

alpha = b1 / (1+b1+b2)
beta = b2 / (1+b1+b2)

print("physical capital coefficient:", b1)
print("human capital coefficient:", b2)
print("Implied alpha:", round(alpha, 4))
print("Implied beta:", round(beta, 4))

# se of alpha and beta
cov_matrix = model_int_Au_r.cov_params().loc[["restricted_k", "restricted_h"], ["restricted_k", "restricted_h"]].to_numpy()
grad_alpha = np.array([(1 + b2) / (1+b1+b2)**2, -b1 / (1+b1+b2)**2])
grad_beta = np.array([-b2 / (1+b1+b2)**2,(1 + b1) / (1+b1+b2)**2])
se_alpha = np.sqrt(grad_alpha @ cov_matrix @ grad_alpha)
se_beta = np.sqrt(grad_beta @ cov_matrix @ grad_beta)

print("se of alpha:", round(se_alpha, 4))
print("se of beta:", round(se_beta, 4))


                            OLS Regression Results                            
Dep. Variable:            ln_GDP_1985   R-squared:                       0.781
Model:                            OLS   Adj. R-squared:                  0.771
Method:                 Least Squares   F-statistic:                     84.25
Date:                Thu, 29 Oct 2026   Prob (F-statistic):           2.44e-23
Time:                        23:58:07   Log-Likelihood:                -45.257
No. Observations:                  75   AIC:                             98.51
Df Residuals:                      71   BIC:                             107.8
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             7.7913      1.192      6.534

In [62]:
# Augumented Solow model
# UM of oecd
oecd = df[df["OECD"] == 1].copy()
y_oecd_Au = oecd["ln_GDP_1985"]
X_oecd_Au = oecd[["ln_investment", "ln_ngd", "ln_school"]]
X_oecd_Au = sm.add_constant(X_oecd_Au)
model_oecd_Au = sm.OLS(y_oecd_Au,X_oecd_Au).fit()
print(model_oecd_Au.summary())

# RM of oecd
oecd["restricted_k"] = (oecd["ln_investment"] - oecd["ln_ngd"])
oecd["restricted_h"] = (oecd["ln_school"] - oecd["ln_ngd"])
y_oecd_Au_r = oecd["ln_GDP_1985"]
X_oecd_Au_r = oecd[["restricted_k", "restricted_h"]]
X_oecd_Au_r = sm.add_constant(X_oecd_Au_r)
model_oecd_Au_r = sm.OLS(y_oecd_Au_r, X_oecd_Au_r).fit()
print(model_oecd_Au_r.summary())

# F test
f_test = model_oecd_Au.f_test("ln_investment + ln_ngd + ln_school = 0")
print(f_test)
print("F-statistic:", float(f_test.fvalue))
print("P-value:", float(f_test.pvalue))

# alpha and beta
b1 = model_oecd_Au_r.params["restricted_k"]
b2 = model_oecd_Au_r.params["restricted_h"]

alpha = b1 / (1+b1+b2)
beta = b2 / (1+b1+b2)

print("physical capital coefficient:", b1)
print("human capital coefficient:", b2)
print("Implied alpha:", round(alpha, 4))
print("Implied beta:", round(beta, 4))

# se of alpha and beta
cov_matrix = model_oecd_Au_r.cov_params().loc[["restricted_k", "restricted_h"], ["restricted_k", "restricted_h"]].to_numpy()
grad_alpha = np.array([(1 + b2) / (1+b1+b2)**2, -b1 / (1+b1+b2)**2])
grad_beta = np.array([-b2 / (1+b1+b2)**2,(1 + b1) / (1+b1+b2)**2])
se_alpha = np.sqrt(grad_alpha @ cov_matrix @ grad_alpha)
se_beta = np.sqrt(grad_beta @ cov_matrix @ grad_beta)

print("se of alpha:", round(se_alpha, 4))
print("se of beta:", round(se_beta, 4))


                            OLS Regression Results                            
Dep. Variable:            ln_GDP_1985   R-squared:                       0.352
Model:                            OLS   Adj. R-squared:                  0.244
Method:                 Least Squares   F-statistic:                     3.264
Date:                Fri, 30 Oct 2026   Prob (F-statistic):             0.0455
Time:                        00:02:20   Log-Likelihood:                -4.6190
No. Observations:                  22   AIC:                             17.24
Df Residuals:                      18   BIC:                             21.60
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             8.6369      2.214      3.901

In [66]:

# Table I 

table1_models = {"Non-oil": {"Unrestricted": model, "Restricted": model_r}, "Intermediate": {"Unrestricted": model_int, "Restricted": model_int_r}, "OECD": {"Unrestricted": model_oecd, "Restricted": model_oecd_r}}

test_model = table1_models["Non-oil"]["Unrestricted"]

print("Coefficients:")
print(test_model.params)

print("Standard Errors:")
print(test_model.bse)

print("Adjusted R-squared:")
print(test_model.rsquared_adj)

print("Number of observations:")
print(test_model.nobs)



Coefficients:
const            5.429883
ln_investment    1.424014
ln_ngd          -1.989774
dtype: float64
Standard Errors:
const            1.583890
ln_investment    0.143106
ln_ngd           0.563362
dtype: float64
Adjusted R-squared:
0.5924619548106453
Number of observations:
98.0
